In [7]:
import os
from dotenv import load_dotenv
from google import genai

load_dotenv()

api_key = os.getenv("GOOGLE_GENERATIVE_AI_API_KEY")

if not api_key:
    raise ValueError("Gemini API key not found in .env")

client = genai.Client(api_key=api_key)

print("Gemini client initialized successfully!")

Gemini client initialized successfully!


In [8]:
def llm(prompt):
    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=prompt
    )
    return response.text

In [9]:
llm("Hey, what's up?")

"Not much! Just hanging out in the digital void, ready to help you with whatever you need. How’s your day going? What's on your mind?"

In [10]:
question = "I just discovered the course. Can I join now?"

answer = llm(question)

print(answer)

To give you the most accurate answer, **I need a little more information.** Because I am an AI, I don’t know which specific course, platform, or institution you are referring to!

To find out if you can still join, please check the following:

### 1. Where did you find the course?
*   **If it’s a platform like Coursera, Udemy, or LinkedIn Learning:** Most of these are "on-demand," meaning you can join and start whenever you want.
*   **If it’s a university or professional certification:** These often have specific "enrollment windows" or start dates. Check the course homepage for an "Enroll" or "Register" button.
*   **If it’s a live cohort-based course:** Check if they have a waitlist for the next session or if they allow late enrollment.

### 2. What to look for on the website:
*   **"Enrollment Closed":** If you see this, you may have to wait for the next term.
*   **"Start Anytime":** This means it is self-paced.
*   **"Syllabus/Schedule":** Look for a calendar. If the course start

In [11]:
context = """
Question: I just discovered the course. Can I still join?

Answer: Yes, but if you want to receive a certificate,
you need to submit your project while we're still
accepting submissions.
"""

In [12]:
prompt = f"""
Your task is to answer questions from course participants
based on the provided context.

Use the context to answer accurately.
If the answer is not found in the context, respond with
"I don't know."

Question:
{question}

Context:
{context}
"""

answer = llm(prompt)
print(answer)

Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.


# Module 1 — Unit 1.3: RAG

## What is RAG?
RAG stands for Retrieval-Augmented Generation.

It combines:
1. Retrieval — finding relevant information.
2. Augmentation — adding that information to the prompt.
3. Generation — using an LLM to generate an answer.

## Why use RAG?
An LLM may not know private, specialized, or up-to-date information. RAG provides relevant external context before generating an answer.

## Basic implementation

- `llm(prompt)` sends a prompt to the LLM and returns text.
- `context` contains the relevant information.
- `question` contains the user's query.
- `prompt` combines the question and context.
- `answer` contains the generated response.

## Manual RAG vs. automatic RAG

Manual: We select and paste the relevant context ourselves.

Automatic: The application searches a knowledge base, retrieves relevant documents, builds a prompt, and sends it to the LLM.

## Important lesson
RAG quality depends heavily on retrieval quality. If the wrong information is retrieved, the generated answer may also be wrong.

## Gemini setup
- Package: `google-genai`
- Client: `genai.Client(...)`
- Generation: `client.models.generate_content(...)`
- Environment variables: load with `load_dotenv()` and read with `os.getenv()`.

Never commit `.env` or expose API keys.

In [13]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"

response = requests.get(docs_url, timeout=30)
response.raise_for_status()

courses_raw = response.json()

print("Number of courses:", len(courses_raw))

Number of courses: 6


In [14]:
courses_raw[0]

{'course': 'stock-markets-analytics-zoomcamp',
 'course_name': 'Stock Markets Analytics Zoomcamp',
 'path': '/json/stock-markets-analytics-zoomcamp.json',
 'questions_count': 93}

In [15]:
documents = []

url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"{url_prefix}{course['path']}"

    course_response = requests.get(course_url, timeout=30)
    course_response.raise_for_status()

    course_data = course_response.json()

    documents.extend(course_data)

print("Total FAQ documents:", len(documents))

Total FAQ documents: 1427


In [16]:
documents[0]

{'id': 'b043d949b7',
 'course': 'stock-markets-analytics-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Do I need to pay for any platform or data for this course?',
 'answer': 'No. All the data sources used in the course (yfinance, pandas_datareader, web scraping) are free, and you do not need a paid GCP account or any other paid service. You are welcome to use paid data if you want, but it is not required.'}

In [17]:
print("Course:", documents[0]["course"])
print("Section:", documents[0]["section"])
print("Question:", documents[0]["question"])
print("Answer:", documents[0]["answer"])

Course: stock-markets-analytics-zoomcamp
Section: General Course-Related Questions
Question: Do I need to pay for any platform or data for this course?
Answer: No. All the data sources used in the course (yfinance, pandas_datareader, web scraping) are free, and you do not need a paid GCP account or any other paid service. You are welcome to use paid data if you want, but it is not required.


# Module 1 — Unit 1.4: The Course FAQ Dataset

## Goal
Load the DataTalks.Club FAQ data to use as our RAG knowledge base.

## Dataset URL
https://datatalks.club/faq/json/courses.json

## Important code

- `import requests` — imports the HTTP requests library.
- `requests.get(url)` — sends a GET request.
- `response.raise_for_status()` — reports HTTP errors.
- `response.json()` — parses JSON into Python objects.
- `for course in courses_raw:` — loops through courses.
- `documents.extend(course_data)` — adds all FAQ entries to a list.
- `documents[0]` — accesses the first document.
- `documents[0]["question"]` — accesses the question field.

## FAQ document fields

- `id` — unique identifier.
- `course` — course slug, used for filtering.
- `section` — category of the question.
- `question` — question text.
- `answer` — answer text.

## RAG connection

1. Download the FAQ data.
2. Combine entries into `documents`.
3. Build a search index in the next lesson.
4. Retrieve relevant FAQ entries for a user's question.
5. Send the retrieved context to Gemini to generate an answer.

## Important concepts

- JSON is a format for structured data exchange.
- Metadata helps organize and filter documents.
- Data preparation is an important part of building RAG applications.
- Use `raise_for_status()` to catch unsuccessful HTTP responses.

In [18]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [19]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '5cc511f85b',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Does the course certificate show the number of course hours?',
  'answer': 'No. The certificate does not 

In [20]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {
        "question": 2.0,
        "section": 0.5
    }

    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [21]:
search_results = search(
    "Can I still join the course?"
)

[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'The homework submission form is still open even though the deadline has passed — can I still submit?',
 'I missed the first homework - can I still get a certificate?',
 'Does the course certificate show the number of course hours?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?']

# Module 1 — Unit 1.5: Search

## Goal
Retrieve relevant FAQ documents for a user's question.

## Lexical search
- Matches words in the query against words in documents.
- Ranks documents according to their relevance.
- Can struggle with synonyms and different wording.

## Semantic search
- Uses embeddings to compare meaning.
- Covered in a later module.

## minsearch

Create an index:

```python
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)
```

Important concepts:
- `Index()` creates the search index.
- `text_fields` are searched and ranked.
- `keyword_fields` support exact filtering.
- `fit()` builds the index from documents.

## Search parameters

- `boost_dict` adjusts the importance of text fields.
- `filter_dict` restricts results using keyword fields.
- `num_results` controls the maximum number of returned results.

Example:

```python
results = index.search(
    "How can I get my certificate?",
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)
```

Display questions:

```python
[doc["question"] for doc in results]
```

## Reusable search function

```python
def search(question, course="llm-zoomcamp"):
    return index.search(
        question,
        boost_dict={"question": 2.0, "section": 0.5},
        filter_dict={"course": course},
        num_results=5
    )
```

## Connection to RAG

1. Search retrieves relevant FAQ documents.
2. Retrieved documents become context.
3. The context and question are added to a prompt.
4. Gemini generates an answer based on that context.

Remember: poor retrieval can lead to poor answers, even when the LLM is capable.

In [22]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [23]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [24]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [25]:
def build_prompt(question, search_results):
    context = build_context(search_results)

    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )

    return prompt.strip()

In [26]:
question = "I just discovered the course. Can I join now?"

search_results = search(question)

prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Does the course certificate show the number of course hours?
A: No. The certificate does not state a total number of hours.

Module 3: Orchestration
Q: Why do we need orchestration / Kestra — can't I just run the code in a notebook?
A: Notebooks are great for learning and experimenting, but real AI w

# Module 1 — Unit 1.6: Building the Prompt

## Goal
Combine the user's question with relevant FAQ search results to create a prompt for the LLM.

## Two prompt components

- INSTRUCTIONS: reusable rules for how the LLM should answer.
- USER_PROMPT_TEMPLATE: contains placeholders for the question and context.

## Build context

```python
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()
```

## Build prompt

```python
def build_prompt(question, search_results):
    context = build_context(search_results)

    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )

    return prompt.strip()
```

## Important Python concepts

- `append()` adds an item to a list.
- `"\n".join(lines)` combines strings using newlines.
- `.strip()` removes leading and trailing whitespace.
- `.format()` replaces placeholders with supplied values.
- A Python function can call another function.

## Connection to RAG

1. Search retrieves relevant documents.
2. `build_context()` formats those documents.
3. `build_prompt()` combines the question and context.
4. The next component sends the prompt and instructions to Gemini.
5. Gemini generates the answer.

## Important lesson
The prompt connects retrieval to generation. Relevant context and clear instructions help ground answers in the knowledge base, but do not guarantee correctness.

In [33]:
response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=f"""
{INSTRUCTIONS}

{prompt}
"""
)

print(response.text)

Yes, you can still join, but if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [32]:
def llm(instructions, user_prompt, model="gemini-3.1-flash-lite"):
    response = client.models.generate_content(
        model=model,
        config={"system_instruction": instructions},
        contents=user_prompt
    )

    return response.text

In [34]:
def rag(query, model="gemini-3.1-flash-lite"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)

    return answer

In [36]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can join now, but if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [37]:
print(rag("How do I get a certificate?"))

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

# Module 1 — Unit 1.7: The LLM

## Goal

Connect the LLM to our existing RAG components so it can generate answers using retrieved FAQ documents.

## 1. The LLM function

```python
def llm(instructions, user_prompt, model="gemini-2.5-flash"):
    response = client.models.generate_content(
        model=model,
        config={"system_instruction": instructions},
        contents=user_prompt
    )
    return response.text
```

- `instructions`: rules for how the model should answer.
- `user_prompt`: question plus retrieved context.
- `model`: model to use; has a default value.
- `response.text`: generated answer.

## 2. The complete RAG function

```python
def rag(query, model="gemini-2.5-flash"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer
```

## 3. RAG workflow

1. Receive the user's question.
2. Search for relevant documents.
3. Build a prompt using the question and retrieved context.
4. Send the prompt and instructions to Gemini.
5. Return the generated answer.

## 4. Important concepts

- **LLM:** Generates text based on its input.
- **Prompt:** Instructions and content sent to the model.
- **Context:** Retrieved information provided to help answer the question.
- **`response.text`:** Accesses Gemini's generated text.
- **`return`:** Sends a function's result back to the code that called it.
- **Default argument:** `model="gemini-2.5-flash"` is used when no model is supplied.
- **Modularity:** Search, prompt building, and generation are separate functions, so one component can be changed without rewriting the entire pipeline.

## 5. Important lesson

RAG combines retrieval with generation. The LLM should use retrieved context rather than relying only on its general knowledge. However, providing context does not guarantee a correct answer; retrieval quality and prompt instructions still matter.

## 6. OpenAI vs Gemini

- Course: `openai_client.responses.create(...)`, then `response.output_text`.
- Our implementation: `client.models.generate_content(...)`, then `response.text`.

The APIs differ, but the overall RAG architecture is the same.

In [1]:

import os
from dotenv import load_dotenv
from google import genai

from ingest import load_faq_data, build_index
from rag_helper import RAGBase

load_dotenv()

api_key = os.getenv("GOOGLE_GENERATIVE_AI_API_KEY")
if not api_key:
    raise ValueError("Gemini API key not found in .env")

client = genai.Client(api_key=api_key)

documents = load_faq_data()
index = build_index(documents)

assistant = RAGBase(
    index=index,
    llm_client=client
)

answer = assistant.rag(
    "I just discovered the course. Can I join now?"
)

print(answer)


Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [2]:
print(assistant.rag("How do I get a certificate?"))
print(assistant.rag("Can I still join the course after it started?"))

To get a certificate, you must follow these requirements:

*   **Participate in a "live" cohort:** You cannot get a certificate in self-paced mode.
*   **Finish a Capstone project:** You must submit your project while a live cohort is accepting them.
*   **Complete peer reviews:** After submitting your project, you must complete the peer reviews assigned to you within the specified peer-review window. 

Homework is not mandatory for obtaining the certificate.
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


# Module 1 — Unit 1.8: RAG Helper

## Goal

Organize the RAG pipeline into reusable Python files.

## Files

- `ingest.py`: downloads FAQ data and creates the search index.
- `rag_helper.py`: contains the RAG class and its methods.
- `classnotes.ipynb`: imports the files and tests the assistant.

## ingest.py

- `load_faq_data()` downloads and combines FAQ documents.
- `build_index(documents)` creates and returns a minsearch index.
- `raise_for_status()` detects failed HTTP requests.
- `extend()` adds multiple items to a list.

## rag_helper.py

`RAGBase` groups the RAG functions and their dependencies.

Methods:

- `__init__()`: stores the index, Gemini client, instructions, template, course, and model.
- `search()`: retrieves relevant documents.
- `build_context()`: formats retrieved documents.
- `build_prompt()`: combines the question and context.
- `llm()`: sends the prompt to Gemini and returns the answer.
- `rag()`: runs the entire RAG pipeline.

## Important Python concepts

- **Class:** Groups related data and methods.
- **Object:** An instance created from a class.
- **`self`:** Refers to the current object.
- **`__init__()`:** Initializes the object.
- **Instance attributes:** Values stored using `self.attribute`.
- **Default arguments:** Allow optional customization.
- **Imports:** Let one Python file reuse code from another.

## Why use a class?

The index and LLM client are passed into the object instead of relying on global variables. This makes the RAG helper easier to reuse and customize.

## Complete RAG workflow

1. Load FAQ data.
2. Build the search index.
3. Search for relevant documents.
4. Build a prompt from the question and context.
5. Send the prompt and instructions to Gemini.
6. Return the generated answer.

## OpenAI vs Gemini

The course uses OpenAI's Responses API. This implementation uses Gemini's `generate_content()` method and `response.text`. The underlying RAG architecture stays the same.

# Module 1 — Part 1 Wrap-up

## What we built

A working Retrieval-Augmented Generation (RAG) system over the DataTalks.Club FAQ dataset, using Gemini for generation.

## RAG pipeline

1. **Retrieve:** Search for relevant documents.
2. **Augment:** Add retrieved context to the question in a prompt.
3. **Generate:** Send the prompt and instructions to the LLM.
4. **Return:** Provide the generated answer to the user.

## Main files and components

- `ingest.py`: loads FAQ documents and builds an in-memory index.
- `rag_helper.py`: implements search, context building, prompt building, LLM calls, and the complete RAG method.
- `sqlite-ingest.ipynb`: prepares and saves the persistent search index.
- `sqlite-query.ipynb`: opens the saved index and runs queries.
- `faq.db`: persistent SQLite search database.
- `classnotes.ipynb`: learning notes and experiments.

## Search backends

- **minsearch:** Lightweight, in-memory text search; the index must be rebuilt after the process ends.
- **sqlitesearch:** Persistent text search backed by SQLite FTS5.
- **Elasticsearch/OpenSearch:** More extensive search capabilities and production scaling options.
- **Vector databases:** Support semantic retrieval using embeddings, among other capabilities.

## RAG vs agents

- Basic RAG follows a predefined flow and commonly searches once per question.
- An agent can decide which tools to use, reformulate queries, and repeat searches based on intermediate results.

## RAG vs fine-tuning

- **RAG:** Retrieves external information at query time; useful for updating knowledge without retraining.
- **Fine-tuning:** Adjusts model weights to influence behavior or specialize a model.

## Main takeaway

Build a reliable RAG baseline first. Then improve retrieval, prompts, data quality, and the model as needed. Choose more advanced infrastructure when the use case justifies it.

# Unit 1.11 — Agents

## 1. Problem with basic RAG
Basic RAG follows a fixed flow:
1. Search the FAQ.
2. Build a prompt using the search results.
3. Send the prompt to the LLM.
4. Return the answer.

Problem: If search results are poor, the pipeline has no built-in recovery. Keyword search can fail with typos or different wording.

## 2. What is an AI agent?
An agent uses an LLM to decide which actions to take and in which order. Instead of always following one fixed workflow, it can choose its next action based on the situation.

An agent may:
- Rewrite a query.
- Correct a typo.
- Search again with different terms.
- Use multiple tools.
- Ask the user a clarifying question.

These abilities require suitable tools and implementation.

## 3. Key concepts
- **Tool / function calling:** The LLM requests that a function, such as `search_faq(query)`, be executed.
- **Agentic loop:** Repeatedly ask the LLM what to do, execute requested tools, return tool results, and continue until a final answer or stopping condition.
- **Agent framework:** A library that helps implement and manage the agent loop and tools.

## 4. Basic RAG vs Agentic RAG
- Basic RAG: a mostly fixed search → prompt → answer workflow.
- Agentic RAG: the LLM can decide how and when to search, potentially retrying or using multiple searches.

**Remember:** A tool-enabled LLM is not automatically a complete agent. The surrounding code must execute tool calls and manage the loop.


# Unit 1.12 — Quick RAG Revision

## Helper files
- `ingest.py`: `load_faq_data()` loads FAQ documents; `build_index()` creates a minsearch index.
- `rag_helper.py`: `RAGBase` handles search, context construction, prompt construction, and the LLM call.

## Setup workflow
1. Load FAQ documents.
2. Build the minsearch index.
3. Create a `RAGBase` assistant.
4. Ask a question using `assistant.rag(question)`.

## Limitation: Keyword search and typos
Example:
- Correct: `How do I run Ollama locally?`
- Typo: `How do I run Olama locally?`

Lexical search may fail to find relevant documents when words are misspelled or phrased differently.

## Why do we need agents?
A fixed RAG pipeline searches once and has no automatic recovery if the results are poor. An agent can decide to correct or rewrite the query, search again, inspect the results, and then answer.

**Key takeaway:** Part 1 built the RAG pipeline. Part 2 makes it more flexible by letting an LLM decide which tools to use and when to use them.
